### Executar este notebook no Jupyter através do navegador

**URL:** *localhost:8888*  
**Senha:** *1234*

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [4]:
spark = SparkSession.builder \
    .appName('spark-test') \
    .master('local[*]') \
    .config("spark.executor.heartbeatInterval", "3600s") \
    .config("spark.network.timeout", "3601s") \
    .getOrCreate()

In [5]:
df = spark.read.csv("data/wc2018-players.csv", inferSchema=True, header=True)

In [6]:
df.show(5)

+---------+---+----+------------------+----------+----------+--------------------+------+------+
|     Team|  #|Pos.| FIFA Popular Name|Birth Date|Shirt Name|                Club|Height|Weight|
+---------+---+----+------------------+----------+----------+--------------------+------+------+
|Argentina|  3|  DF|TAGLIAFICO Nicolas|31.08.1992|TAGLIAFICO|      AFC Ajax (NED)|   169|    65|
|Argentina| 22|  MF|    PAVON Cristian|21.01.1996|     PAVÓN|CA Boca Juniors (...|   169|    65|
|Argentina| 15|  MF|    LANZINI Manuel|15.02.1993|   LANZINI|West Ham United F...|   167|    66|
|Argentina| 18|  DF|    SALVIO Eduardo|13.07.1990|    SALVIO|    SL Benfica (POR)|   167|    69|
|Argentina| 10|  FW|      MESSI Lionel|24.06.1987|     MESSI|  FC Barcelona (ESP)|   170|    72|
+---------+---+----+------------------+----------+----------+--------------------+------+------+
only showing top 5 rows


In [7]:
df.printSchema()

root
 |-- Team: string (nullable = true)
 |-- #: integer (nullable = true)
 |-- Pos.: string (nullable = true)
 |-- FIFA Popular Name: string (nullable = true)
 |-- Birth Date: string (nullable = true)
 |-- Shirt Name: string (nullable = true)
 |-- Club: string (nullable = true)
 |-- Height: integer (nullable = true)
 |-- Weight: integer (nullable = true)



In [8]:
# Como a coluna "Pos." tem um ., pode dar erro em algumas funcoes, por isso é necessário alterar o nome
# Pode alterar quantas colunas quiser em sequencia

df = df.withColumnRenamed('Pos.', 'pos')

In [9]:
import string
def removePontuaction(text: str) -> str:
    table = str.maketrans('', '', string.punctuation)

    if text == '#':
        return 'num'
    
    result = text.translate(table)
    result = result.replace(' ', '_').strip()
    result = result.lower()
    return result    

In [10]:
for column in df.columns:
    df = df.withColumnRenamed(column, removePontuaction(column))

In [11]:
df.printSchema()

root
 |-- team: string (nullable = true)
 |-- num: integer (nullable = true)
 |-- pos: string (nullable = true)
 |-- fifa_popular_name: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- shirt_name: string (nullable = true)
 |-- club: string (nullable = true)
 |-- height: integer (nullable = true)
 |-- weight: integer (nullable = true)



In [12]:
# Se for dataframe pequeno

df.toPandas().isna().sum()

team                 0
num                  0
pos                  0
fifa_popular_name    0
birth_date           0
shirt_name           0
club                 0
height               0
weight               0
dtype: int64

In [13]:
# Se for dataframe grande

for column in df.columns:
    print(column, df.filter(df[column].isNull()).count())

team 0
num 0
pos 0
fifa_popular_name 0
birth_date 0
shirt_name 0
club 0
height 0
weight 0


In [14]:
df.select('team', 'fifa_popular_name').show(5)

+---------+------------------+
|     team| fifa_popular_name|
+---------+------------------+
|Argentina|TAGLIAFICO Nicolas|
|Argentina|    PAVON Cristian|
|Argentina|    LANZINI Manuel|
|Argentina|    SALVIO Eduardo|
|Argentina|      MESSI Lionel|
+---------+------------------+
only showing top 5 rows


In [15]:
df.select(col('team'), col('height')).show(5)

+---------+------+
|     team|height|
+---------+------+
|Argentina|   169|
|Argentina|   169|
|Argentina|   167|
|Argentina|   167|
|Argentina|   170|
+---------+------+
only showing top 5 rows


In [16]:
df.select(df['team']).show(5)

+---------+
|     team|
+---------+
|Argentina|
|Argentina|
|Argentina|
|Argentina|
|Argentina|
+---------+
only showing top 5 rows


In [17]:
df.select(col('team').alias('time')).show(5)

+---------+
|     time|
+---------+
|Argentina|
|Argentina|
|Argentina|
|Argentina|
|Argentina|
+---------+
only showing top 5 rows


In [18]:
df.select(df['team'].alias('time')).show(5)

+---------+
|     time|
+---------+
|Argentina|
|Argentina|
|Argentina|
|Argentina|
|Argentina|
+---------+
only showing top 5 rows


In [19]:
df.select('team fifa_popular_name height'.split()).show(5)

+---------+------------------+------+
|     team| fifa_popular_name|height|
+---------+------------------+------+
|Argentina|TAGLIAFICO Nicolas|   169|
|Argentina|    PAVON Cristian|   169|
|Argentina|    LANZINI Manuel|   167|
|Argentina|    SALVIO Eduardo|   167|
|Argentina|      MESSI Lionel|   170|
+---------+------------------+------+
only showing top 5 rows


In [20]:
df.filter('team = "Brazil"').show()

+------+---+---+-----------------+----------+-----------+--------------------+------+------+
|  team|num|pos|fifa_popular_name|birth_date| shirt_name|                club|height|weight|
+------+---+---+-----------------+----------+-----------+--------------------+------+------+
|Brazil| 18| MF|             FRED|05.03.1993|       FRED|FC Shakhtar Donet...|   169|    64|
|Brazil| 21| FW|           TAISON|13.01.1988|     TAISON|FC Shakhtar Donet...|   172|    64|
|Brazil| 17| MF|      FERNANDINHO|04.05.1985|FERNANDINHO|Manchester City F...|   179|    67|
|Brazil| 22| DF|           FAGNER|11.06.1989|     FAGNER|SC Corinthians (BRA)|   168|    67|
|Brazil| 10| FW|           NEYMAR|05.02.1992|  NEYMAR JR|Paris Saint-Germa...|   175|    68|
|Brazil| 11| MF|PHILIPPE COUTINHO|12.06.1992|P. COUTINHO|  FC Barcelona (ESP)|   172|    68|
|Brazil|  7| FW|    DOUGLAS COSTA|14.09.1990|   D. COSTA|   Juventus FC (ITA)|   182|    70|
|Brazil|  6| DF|      FILIPE LUIS|09.08.1985|FILIPE LUIS|Atletico Madr

In [21]:
df.filter(col('shirt_name') == 'FRED').show()

+------+---+---+-----------------+----------+----------+--------------------+------+------+
|  team|num|pos|fifa_popular_name|birth_date|shirt_name|                club|height|weight|
+------+---+---+-----------------+----------+----------+--------------------+------+------+
|Brazil| 18| MF|             FRED|05.03.1993|      FRED|FC Shakhtar Donet...|   169|    64|
+------+---+---+-----------------+----------+----------+--------------------+------+------+



In [22]:
df.filter((col('team') == 'Argentina') & (col('height') > 180)).show()

+---------+---+---+------------------+----------+----------+--------------------+------+------+
|     team|num|pos| fifa_popular_name|birth_date|shirt_name|                club|height|weight|
+---------+---+---+------------------+----------+----------+--------------------+------+------+
|Argentina|  4| DF|  ANSALDI Cristian|20.09.1986|   ANSALDI|     Torino FC (ITA)|   181|    73|
|Argentina|  9| FW|   HIGUAIN Gonzalo|10.12.1987|   HIGUAÍN|   Juventus FC (ITA)|   184|    75|
|Argentina| 23| GK|CABALLERO Wilfredo|28.09.1981| CABALLERO|    Chelsea FC (ENG)|   186|    80|
|Argentina|  2| DF|   MERCADO Gabriel|18.03.1987|   MERCADO|    Sevilla FC (ESP)|   181|    81|
|Argentina| 17| DF|  OTAMENDI Nicolas|12.02.1988|  OTAMENDI|Manchester City F...|   181|    81|
|Argentina| 16| DF|       ROJO Marcos|20.03.1990|      ROJO|Manchester United...|   189|    82|
|Argentina|  6| DF|    FAZIO Federico|17.03.1987|     FAZIO|       AS Roma (ITA)|   199|    85|
|Argentina| 12| GK|     ARMANI Franco|16

In [23]:
df.filter((col('shirt_name') == 'FRED') | (col('shirt_name') == 'FERNANDINHO')).show()

+------+---+---+-----------------+----------+-----------+--------------------+------+------+
|  team|num|pos|fifa_popular_name|birth_date| shirt_name|                club|height|weight|
+------+---+---+-----------------+----------+-----------+--------------------+------+------+
|Brazil| 18| MF|             FRED|05.03.1993|       FRED|FC Shakhtar Donet...|   169|    64|
|Brazil| 17| MF|      FERNANDINHO|04.05.1985|FERNANDINHO|Manchester City F...|   179|    67|
+------+---+---+-----------------+----------+-----------+--------------------+------+------+



In [24]:
df.filter((col('shirt_name').like('FR%'))).show()

+-----------+---+---+------------------+----------+-----------+--------------------+------+------+
|       team|num|pos| fifa_popular_name|birth_date| shirt_name|                club|height|weight|
+-----------+---+---+------------------+----------+-----------+--------------------+------+------+
|     Brazil| 18| MF|              FRED|05.03.1993|       FRED|FC Shakhtar Donet...|   169|    64|
|    Iceland|  3| MF|FRIDJONSSON Samuel|22.02.1996|FRIDJÓNSSON|  Valerenga IF (NOR)|   185|    78|
|Switzerland|  8| MF|      FREULER Remo|15.04.1992|    FREULER|Atalanta Bergamo ...|   181|    75|
+-----------+---+---+------------------+----------+-----------+--------------------+------+------+



In [25]:
df = df.withColumn('world_cup', lit(2018))
df.show(5)

+---------+---+---+------------------+----------+----------+--------------------+------+------+---------+
|     team|num|pos| fifa_popular_name|birth_date|shirt_name|                club|height|weight|world_cup|
+---------+---+---+------------------+----------+----------+--------------------+------+------+---------+
|Argentina|  3| DF|TAGLIAFICO Nicolas|31.08.1992|TAGLIAFICO|      AFC Ajax (NED)|   169|    65|     2018|
|Argentina| 22| MF|    PAVON Cristian|21.01.1996|     PAVÓN|CA Boca Juniors (...|   169|    65|     2018|
|Argentina| 15| MF|    LANZINI Manuel|15.02.1993|   LANZINI|West Ham United F...|   167|    66|     2018|
|Argentina| 18| DF|    SALVIO Eduardo|13.07.1990|    SALVIO|    SL Benfica (POR)|   167|    69|     2018|
|Argentina| 10| FW|      MESSI Lionel|24.06.1987|     MESSI|  FC Barcelona (ESP)|   170|    72|     2018|
+---------+---+---+------------------+----------+----------+--------------------+------+------+---------+
only showing top 5 rows


In [26]:
df = df.withColumn('sub', substring('team', 0, 3))
df.show()

+---------+---+---+------------------+----------+----------+--------------------+------+------+---------+---+
|     team|num|pos| fifa_popular_name|birth_date|shirt_name|                club|height|weight|world_cup|sub|
+---------+---+---+------------------+----------+----------+--------------------+------+------+---------+---+
|Argentina|  3| DF|TAGLIAFICO Nicolas|31.08.1992|TAGLIAFICO|      AFC Ajax (NED)|   169|    65|     2018|Arg|
|Argentina| 22| MF|    PAVON Cristian|21.01.1996|     PAVÓN|CA Boca Juniors (...|   169|    65|     2018|Arg|
|Argentina| 15| MF|    LANZINI Manuel|15.02.1993|   LANZINI|West Ham United F...|   167|    66|     2018|Arg|
|Argentina| 18| DF|    SALVIO Eduardo|13.07.1990|    SALVIO|    SL Benfica (POR)|   167|    69|     2018|Arg|
|Argentina| 10| FW|      MESSI Lionel|24.06.1987|     MESSI|  FC Barcelona (ESP)|   170|    72|     2018|Arg|
|Argentina|  4| DF|  ANSALDI Cristian|20.09.1986|   ANSALDI|     Torino FC (ITA)|   181|    73|     2018|Arg|
|Argentina

In [27]:
df = df.withColumn('birth_year', substring(col('birth_date'), -4, 4))
df.show()

+---------+---+---+------------------+----------+----------+--------------------+------+------+---------+---+----------+
|     team|num|pos| fifa_popular_name|birth_date|shirt_name|                club|height|weight|world_cup|sub|birth_year|
+---------+---+---+------------------+----------+----------+--------------------+------+------+---------+---+----------+
|Argentina|  3| DF|TAGLIAFICO Nicolas|31.08.1992|TAGLIAFICO|      AFC Ajax (NED)|   169|    65|     2018|Arg|      1992|
|Argentina| 22| MF|    PAVON Cristian|21.01.1996|     PAVÓN|CA Boca Juniors (...|   169|    65|     2018|Arg|      1996|
|Argentina| 15| MF|    LANZINI Manuel|15.02.1993|   LANZINI|West Ham United F...|   167|    66|     2018|Arg|      1993|
|Argentina| 18| DF|    SALVIO Eduardo|13.07.1990|    SALVIO|    SL Benfica (POR)|   167|    69|     2018|Arg|      1990|
|Argentina| 10| FW|      MESSI Lionel|24.06.1987|     MESSI|  FC Barcelona (ESP)|   170|    72|     2018|Arg|      1987|
|Argentina|  4| DF|  ANSALDI Cri

In [28]:
df.withColumn('Concat', concat('team', 'shirt_name')).show()

+---------+---+---+------------------+----------+----------+--------------------+------+------+---------+---+----------+-------------------+
|     team|num|pos| fifa_popular_name|birth_date|shirt_name|                club|height|weight|world_cup|sub|birth_year|             Concat|
+---------+---+---+------------------+----------+----------+--------------------+------+------+---------+---+----------+-------------------+
|Argentina|  3| DF|TAGLIAFICO Nicolas|31.08.1992|TAGLIAFICO|      AFC Ajax (NED)|   169|    65|     2018|Arg|      1992|ArgentinaTAGLIAFICO|
|Argentina| 22| MF|    PAVON Cristian|21.01.1996|     PAVÓN|CA Boca Juniors (...|   169|    65|     2018|Arg|      1996|     ArgentinaPAVÓN|
|Argentina| 15| MF|    LANZINI Manuel|15.02.1993|   LANZINI|West Ham United F...|   167|    66|     2018|Arg|      1993|   ArgentinaLANZINI|
|Argentina| 18| DF|    SALVIO Eduardo|13.07.1990|    SALVIO|    SL Benfica (POR)|   167|    69|     2018|Arg|      1990|    ArgentinaSALVIO|
|Argentina| 1

In [29]:
df.withColumn('concat', concat_ws(' - ', 'team', 'shirt_name')).show()

+---------+---+---+------------------+----------+----------+--------------------+------+------+---------+---+----------+--------------------+
|     team|num|pos| fifa_popular_name|birth_date|shirt_name|                club|height|weight|world_cup|sub|birth_year|              concat|
+---------+---+---+------------------+----------+----------+--------------------+------+------+---------+---+----------+--------------------+
|Argentina|  3| DF|TAGLIAFICO Nicolas|31.08.1992|TAGLIAFICO|      AFC Ajax (NED)|   169|    65|     2018|Arg|      1992|Argentina - TAGLI...|
|Argentina| 22| MF|    PAVON Cristian|21.01.1996|     PAVÓN|CA Boca Juniors (...|   169|    65|     2018|Arg|      1996|   Argentina - PAVÓN|
|Argentina| 15| MF|    LANZINI Manuel|15.02.1993|   LANZINI|West Ham United F...|   167|    66|     2018|Arg|      1993| Argentina - LANZINI|
|Argentina| 18| DF|    SALVIO Eduardo|13.07.1990|    SALVIO|    SL Benfica (POR)|   167|    69|     2018|Arg|      1990|  Argentina - SALVIO|
|Argen

In [30]:
df.printSchema()

root
 |-- team: string (nullable = true)
 |-- num: integer (nullable = true)
 |-- pos: string (nullable = true)
 |-- fifa_popular_name: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- shirt_name: string (nullable = true)
 |-- club: string (nullable = true)
 |-- height: integer (nullable = true)
 |-- weight: integer (nullable = true)
 |-- world_cup: integer (nullable = false)
 |-- sub: string (nullable = true)
 |-- birth_year: string (nullable = true)



In [31]:
df = df.withColumn('birth_year', col('birth_year').cast(IntegerType()))
df.printSchema()

root
 |-- team: string (nullable = true)
 |-- num: integer (nullable = true)
 |-- pos: string (nullable = true)
 |-- fifa_popular_name: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- shirt_name: string (nullable = true)
 |-- club: string (nullable = true)
 |-- height: integer (nullable = true)
 |-- weight: integer (nullable = true)
 |-- world_cup: integer (nullable = false)
 |-- sub: string (nullable = true)
 |-- birth_year: integer (nullable = true)



In [32]:
day = udf(lambda x: x.split('.')[0])
month = udf(lambda x: x.split('.')[1])

In [34]:
df = df.withColumn('birth_day', day(col('birth_date')).cast(IntegerType()))
df = df.withColumn('birth_month', month(col('birth_date')).cast(IntegerType()))
df = df.withColumn('birth_date', concat_ws('-', 'birth_year', 'birth_month', 'birth_day'))
df = df.withColumn('birth_date', col('birth_date').cast(DateType()))
df.show()

Py4JJavaError: An error occurred while calling o285.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 48.0 failed 1 times, most recent failure: Lost task 0.0 in stage 48.0 (TID 39) (Lucas-Desktop executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:281)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:154)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:309)
	at org.apache.spark.sql.execution.python.BatchEvalPythonEvaluatorFactory.evaluate(BatchEvalPythonExec.scala:95)
	at org.apache.spark.sql.execution.python.EvalPythonEvaluatorFactory$EvalPythonPartitionEvaluator.eval(EvalPythonEvaluatorFactory.scala:113)
	at org.apache.spark.sql.execution.python.EvalPythonExec.$anonfun

In [ ]:
df.printSchema()

root
 |-- team: string (nullable = true)
 |-- num: integer (nullable = true)
 |-- pos: string (nullable = true)
 |-- fifa_popular_name: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- shirt_name: string (nullable = true)
 |-- club: string (nullable = true)
 |-- height: integer (nullable = true)
 |-- weight: integer (nullable = true)
 |-- world_cup: integer (nullable = false)
 |-- sub: string (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_day: integer (nullable = true)
 |-- birth_month: integer (nullable = true)

